In [33]:

import json
import os
import pandas as pd
from pathlib import Path
from openai import OpenAI


In [34]:

# ── Paths ─────────────────────────────────────────────────────────────────────

ROOT      = Path("../")
JSON_PATH = ROOT / "data" / "yt_data" / "dance_youtube_data.json"
CSV_PATH  = ROOT / "data" / "yt_data" / "dance_videos.csv"
KEY_PATH  = ROOT / "data" / "GPT4oKEY.txt"

# OpenRouter key — compatible with the openai SDK via base_url override
api_key = KEY_PATH.read_text(encoding="utf-8").strip()
client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
)
MODEL = "openai/gpt-4o-mini"
print(f"Client ready → {MODEL}")

Client ready → openai/gpt-4o-mini


In [35]:

# ── Load comments from JSON ────────────────────────────────────────────────────
with JSON_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

# Flatten into a list of {video_id, text} dicts
comment_rows = []
for entry in data.values():
    for video in entry.get("videos", []):
        video_id = (
            video.get("basic_info", {}).get("video_id")
            or video.get("comprehensive_info", {}).get("video_id")
        )
        for comment in video.get("comments", []):
            text = str(comment.get("text", "")).strip()
            if text and video_id:
                comment_rows.append({"video_id": video_id, "text": text})

comments_df = pd.DataFrame(comment_rows)
print(f"Total comments loaded: {len(comments_df)}")
print(f"Unique videos:         {comments_df['video_id'].nunique()}")
comments_df.head(3)

Total comments loaded: 3757
Unique videos:         403


,video_id,text
0,LH_EyxsupRE,Learning tap was my secret childhood dream. I ...
1,LH_EyxsupRE,Me at 3PM: I should really spend my night stud...
2,LH_EyxsupRE,Wow... i took a class and i learned more in th...


In [36]:

# ── Sentiment function using OpenRouter ───────────────────────────────────────

def score_comment(text: str) -> tuple[str, float]:
    """Returns (label, score) for a single comment. label: POSITIVE/NEUTRAL/NEGATIVE, score: 1/0/-1."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Classify the sentiment of a YouTube comment about a dance video. "
                    "Reply with exactly one word: POSITIVE, NEUTRAL, or NEGATIVE."
                ),
            },
            {"role": "user", "content": text},
        ],
    )
    label = response.choices[0].message.content.strip().upper()
    if label not in ("POSITIVE", "NEUTRAL", "NEGATIVE"):
        label = "NEUTRAL"
    score_map = {"POSITIVE": 1.0, "NEUTRAL": 0.0, "NEGATIVE": -1.0}
    return label, score_map[label]

print("Sentiment function ready.")

Sentiment function ready.


In [37]:

# ── Run sentiment on every comment ────────────────────────────────────────────

sentiments = []
texts = comments_df["text"].tolist()

for idx, text in enumerate(texts):
    label, score = score_comment(text)
    sentiments.append({"label": label, "score": score})

    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} / {len(texts)} comments...")

print(f"\nDone — {len(texts)} comments processed.")

comments_df["sentiment_label"] = [s["label"] for s in sentiments]
comments_df["score"]           = [s["score"] for s in sentiments]
comments_df.head(5)

Processed 10 / 3757 comments...
Processed 20 / 3757 comments...
Processed 30 / 3757 comments...
Processed 40 / 3757 comments...
Processed 50 / 3757 comments...
Processed 60 / 3757 comments...
Processed 70 / 3757 comments...
Processed 80 / 3757 comments...
Processed 90 / 3757 comments...
Processed 100 / 3757 comments...
Processed 110 / 3757 comments...
Processed 120 / 3757 comments...
Processed 130 / 3757 comments...
Processed 140 / 3757 comments...
Processed 150 / 3757 comments...
Processed 160 / 3757 comments...
Processed 170 / 3757 comments...
Processed 180 / 3757 comments...
Processed 190 / 3757 comments...
Processed 200 / 3757 comments...
Processed 210 / 3757 comments...
Processed 220 / 3757 comments...
Processed 230 / 3757 comments...
Processed 240 / 3757 comments...
Processed 250 / 3757 comments...
Processed 260 / 3757 comments...
Processed 270 / 3757 comments...
Processed 280 / 3757 comments...
Processed 290 / 3757 comments...
Processed 300 / 3757 comments...
Processed 310 / 375

,video_id,text,sentiment_label,score
0,LH_EyxsupRE,Learning tap was my secret childhood dream. I ...,POSITIVE,1.0
1,LH_EyxsupRE,Me at 3PM: I should really spend my night stud...,NEUTRAL,0.0
2,LH_EyxsupRE,Wow... i took a class and i learned more in th...,POSITIVE,1.0
3,LH_EyxsupRE,who's here to learn how to tap bc they're bore...,NEUTRAL,0.0
4,LH_EyxsupRE,Great for beginners and advanced tappers who h...,POSITIVE,1.0


In [38]:

# ── Aggregate per video ────────────────────────────────────────────────────────

def majority_label(labels):
    """Return the most frequent label, lowercased to match the existing CSV format."""
    return labels.value_counts().idxmax().lower()

agg = (
    comments_df
    .groupby("video_id")
    .agg(
        comment_sentiment               = ("sentiment_label", majority_label),
        comment_sentiment_avg_compound  = ("score", lambda x: round(x.mean(), 4)),
        comment_sentiment_comment_count = ("text", "count"),
    )
    .reset_index()
)

print(f"Aggregated sentiment for {len(agg)} videos.")
agg.head()

Aggregated sentiment for 403 videos.


,video_id,comment_sentiment,comment_sentiment_avg_compound,comment_sentiment_comment_count
0,--J5QVgWZqg,positive,0.9000,10
1,-3wwTJg6z1I,positive,0.5000,2
2,-I5CTGc93mk,positive,0.6667,3
3,-UqAnPyFK4M,positive,1.0000,10
4,-ftaSOJDRbg,positive,1.0000,10


In [39]:

# ── Merge into dance_videos.csv ────────────────────────────────────────────────
videos_df = pd.read_csv(CSV_PATH)
print(f"CSV rows before merge: {len(videos_df)}")

# Drop old sentiment columns if they already exist
drop_cols = [
    "comment_sentiment",
    "comment_sentiment_avg_compound",
    "comment_sentiment_comment_count",
]
videos_df = videos_df.drop(columns=[c for c in drop_cols if c in videos_df.columns])

merged = videos_df.merge(agg, on="video_id", how="left")

merged.to_csv(CSV_PATH, index=False, encoding="utf-8")
print(f"CSV saved  → {CSV_PATH}")
print(f"Total rows : {len(merged)} | Videos with sentiment: {agg['video_id'].nunique()}")
merged[["video_id", "comment_sentiment", "comment_sentiment_avg_compound",
        "comment_sentiment_comment_count"]].dropna().head(10)


CSV rows before merge: 527
CSV saved  → ../data/yt_data/dance_videos.csv
Total rows : 527 | Videos with sentiment: 403


,video_id,comment_sentiment,comment_sentiment_avg_compound,comment_sentiment_comment_count
0,LH_EyxsupRE,positive,0.7,20.0
1,HZruz8wO0g8,positive,0.9,10.0
2,OSa4tPmduk8,positive,0.9,10.0
3,LH_EyxsupRE,positive,0.7,20.0
4,dX7Tz_VBfK0,positive,0.8,10.0
5,B-8kxNao4aw,positive,1.0,10.0
6,QGm6v9-1ZBU,positive,0.4,10.0
7,ozagJ6DYDUA,positive,0.6,10.0
8,C9_u5Eg5ClI,positive,1.0,4.0
9,6ur6ge5fvk0,neutral,0.2,30.0
